# 02 — Augmentation Debug

Saves 20 augmented samples to `/tmp/aug_debug/` and lets you verify that
bounding boxes and polygon vertices align correctly with image content after
each augmentation stage.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path

tf.config.run_functions_eagerly(True)
OUTPUT_DIR = Path('/tmp/aug_debug')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('TF:', tf.__version__, '| output:', OUTPUT_DIR)

## 1  Synthetic example

Build a realistic decoded dict without needing TFDS.

In [ ]:
rng = np.random.RandomState(42)
H, W = 480, 640

# Background image: random gradient
img = rng.randint(50, 200, (H, W, 3), dtype=np.uint8)
img_tf = tf.constant(img)

# 3 ground-truth boxes: yxyx normalised
boxes_np = np.array([
    [0.10, 0.05, 0.40, 0.35],
    [0.55, 0.45, 0.85, 0.80],
    [0.20, 0.60, 0.50, 0.90],
], dtype=np.float32)
boxes_tf = tf.constant(boxes_np)

# Polygon for each box: a rough ellipse, stored as flat xy pairs
MAX_V = 40  # 20 vertices × 2
def make_ellipse_poly(box, n_verts=10):
    y0, x0, y1, x1 = box
    cy, cx = (y0+y1)/2, (x0+x1)/2
    ry, rx = (y1-y0)/2 * 0.8, (x1-x0)/2 * 0.8
    pts = []
    for i in range(n_verts):
        a = 2 * np.pi * i / n_verts
        pts += [cx + rx*np.cos(a), cy + ry*np.sin(a)]
    # Pad with -1
    pts += [-1.0] * (MAX_V - len(pts))
    return np.array(pts, dtype=np.float32)

polys_np = np.stack([make_ellipse_poly(b) for b in boxes_np])
polys_tf = tf.constant(polys_np)
print('image:', img_tf.shape, 'boxes:', boxes_tf.shape, 'polys:', polys_tf.shape)

## 2  Helper: visualise boxes + polygon vertices

In [ ]:
def draw_and_save(img_np, boxes_np, polys_np, title, path):
    """Draw bboxes and polygon vertices on image and save to *path*."""
    h, w = img_np.shape[:2]
    fig, ax = plt.subplots(1, 1, figsize=(8, 6))
    ax.imshow(img_np)
    ax.set_title(title)

    colours = ['red', 'lime', 'cyan']
    for i, (box, poly) in enumerate(zip(boxes_np, polys_np)):
        col = colours[i % len(colours)]
        y0, x0, y1, x1 = box
        rect = patches.Rectangle(
            (x0*w, y0*h), (x1-x0)*w, (y1-y0)*h,
            linewidth=2, edgecolor=col, facecolor='none'
        )
        ax.add_patch(rect)
        # Plot valid polygon vertices (xy pairs)
        pts = poly.reshape(-1, 2)
        valid = pts[:, 0] >= 0
        vx = pts[valid, 0] * w
        vy = pts[valid, 1] * h
        ax.scatter(vx, vy, s=15, c=col, zorder=5)

    fig.tight_layout()
    fig.savefig(path, dpi=100)
    plt.close(fig)

draw_and_save(
    img, boxes_np, polys_np,
    'Original',
    OUTPUT_DIR / '00_original.png'
)
print('saved', OUTPUT_DIR / '00_original.png')

## 3  Test each augmentation stage

In [ ]:
from data_pipeline.augmentations import (
    random_horizontal_flip, random_affine, hsv_augment,
    clip_boxes, clip_polygon_coords, apply_albumentations
)

# ---- Horizontal flip ----
img_f, boxes_f, polys_f = random_horizontal_flip(img_tf, boxes_tf, polys_tf, MAX_V)
draw_and_save(
    img_f.numpy(), boxes_f.numpy(), polys_f.numpy(),
    'After horizontal flip',
    OUTPUT_DIR / '01_flip.png'
)
print('saved 01_flip.png')

In [ ]:
# ---- Affine (scale + translate) ----
img_a, boxes_a, polys_a = random_affine(
    img_tf, boxes_tf, polys_tf,
    translate=0.1, scale_min=0.7, scale_max=1.3,
    output_size=[480, 640],
)
draw_and_save(
    img_a.numpy(), boxes_a.numpy(), polys_a.numpy(),
    'After affine (scale+translate)',
    OUTPUT_DIR / '02_affine.png'
)
print('saved 02_affine.png')

In [ ]:
# ---- HSV ----
img_norm = tf.cast(img_tf, tf.float32) / 255.0
img_hsv  = hsv_augment(img_norm, hue=0.1, sat=0.5, val=0.3).numpy()
draw_and_save(
    (img_hsv * 255).astype(np.uint8), boxes_np, polys_np,
    'After HSV augmentation',
    OUTPUT_DIR / '03_hsv.png'
)
print('saved 03_hsv.png')

In [ ]:
# ---- Albumentations ----
img_alb = apply_albumentations(img_norm, freq=1.0).numpy()
draw_and_save(
    (img_alb * 255).astype(np.uint8), boxes_np, polys_np,
    'After Albumentations',
    OUTPUT_DIR / '04_albumentations.png'
)
print('saved 04_albumentations.png')

## 4  Test PolyYOLO polygon preprocessing

In [ ]:
from data_pipeline.yolo_parser import V8ParserExtended

parser = V8ParserExtended(
    output_size=[672, 672],
    expanded_strides={'3': 8, '4': 16, '5': 32},
    levels=['3', '4', '5'],
    max_vertices=MAX_V,
    angle_step=15,
    with_polygons=True,
)

poly_yolo = parser._preprocess_polygons_v2(boxes_tf, polys_tf, angle_step=15)
print('PolyYOLO output shape:', poly_yolo.shape)  # [3, 72]

# Check confidence sums (each instance should have some non-zero confs)
poly_np = poly_yolo.numpy()
for i in range(len(boxes_np)):
    confs = poly_np[i, 2::3]  # every 3rd value starting from index 2
    print(f'  Instance {i}: {int(confs.sum())} / 24 bins assigned')

## 5  Full parser round-trip

In [ ]:
decoded = {
    'image':                 img_tf,
    'groundtruth_boxes':     boxes_tf,
    'groundtruth_classes':   tf.constant([0, 1, 2], dtype=tf.int64),
    'groundtruth_polygons':  polys_tf,
    'groundtruth_is_crowd':  tf.constant([False, False, False]),
    'groundtruth_area':      tf.constant([500.0, 800.0, 600.0]),
    'groundtruth_dontcare':  tf.zeros([3], tf.int64),
    'source_id':             tf.constant('debug_001'),
}

train_fn = parser.parse_fn(is_training=True)
img_out, labels = train_fn(decoded)

print('image shape:    ', img_out.shape)
print('bbox shape:     ', labels['bbox'].shape)
print('classes shape:  ', labels['classes'].shape)
print('polygons shape: ', labels['polygons'].shape)
print('n_gt:           ', labels['n_gt'].numpy())
print('ignore_bg:      ', labels['ignore_bg'].numpy())

# Draw augmented result
n = int(labels['n_gt'].numpy())
img_arr = (img_out.numpy() * 255).astype(np.uint8)
boxes_v = labels['bbox'].numpy()[:n]

# Back-compute polygon vertices from PolyYOLO for visualisation
def polyyolo_to_pts(box, pv, img_h, img_w):
    cy = (box[0]+box[2])/2; cx = (box[1]+box[3])/2
    xy = []
    for b in range(24):
        dx, dy, conf = pv[b*3], pv[b*3+1], pv[b*3+2]
        if conf > 0.5:
            xy.append([cx + dx, cy + dy])
    return np.array(xy) if xy else np.zeros((0, 2))

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(img_arr)
ax.set_title('After full training parse (PolyYOLO vertices)')
h_out, w_out = img_arr.shape[:2]
for i, box in enumerate(boxes_v):
    col = ['red','lime','cyan'][i % 3]
    y0, x0, y1, x1 = box
    rect = patches.Rectangle((x0*w_out, y0*h_out), (x1-x0)*w_out, (y1-y0)*h_out,
                               linewidth=2, edgecolor=col, facecolor='none')
    ax.add_patch(rect)
    pts = polyyolo_to_pts(box, labels['polygons'].numpy()[i], h_out, w_out)
    if len(pts):
        ax.scatter(pts[:,0]*w_out, pts[:,1]*h_out, s=20, c=col)

fig.tight_layout()
fig.savefig(OUTPUT_DIR / '05_parser_output.png', dpi=100)
plt.show()
print('saved 05_parser_output.png')

In [ ]:
# Summary: list saved files
print('\nSaved augmentation debug images:')
for p in sorted(OUTPUT_DIR.glob('*.png')):
    print(' ', p)